In [120]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
from models.nn import MLP
from config import config
from work_with_data import work
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV, ElasticNet
from sklearn.svm import LinearSVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor, VotingRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.neighbors import KNeighborsRegressor
import shap
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.model_selection import train_test_split
import optuna
from sklearn.model_selection import KFold

In [95]:
prep = work()

In [96]:
# import data
FE = config.FE
use_submit = config.use_submit
df = prep.from_csv()
X_train, y = prep.forward(df = df, FE = FE)
if config.NN.on:
    X_train_nn, y_nn, input_feat = prep.forward(df = df, FE = FE, nn = config.NN.on)
df_test = prep.from_csv(train = False)
if config.NN.on:
    X_test_nn, _, input_feat = prep.forward(df = df_test, FE = FE, nn = config.NN.on, use_submit=use_submit)
X_test, _ = prep.forward(df = df_test, use_submit = use_submit, FE = FE)

In [133]:
def objective(trial):
    param_cat = {
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 1, log=True),
        'depth': trial.suggest_int('depth', 4, 7),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10, log=True),
    }
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    mses = []
    
    for train_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = CatBoostRegressor(**param_cat,
                    early_stopping_rounds=50,
                    iterations=4000,
                    loss_function='RMSE',
                    verbose=False,
                    bootstrap_type='Bayesian'
                )

        model.fit(
            X_tr, y_tr,
        )

        preds = model.predict(X_val)
        mses.append(root_mean_squared_error(np.exp(y_val), np.exp(preds)))
    
    return np.mean(mses)

In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(func=objective, n_trials=10)

In [64]:
masker = shap.maskers.Independent(data=X_train)
explainer = shap.LinearExplainer(lasso, masker)
shap_values = explainer.shap_values(X = X_train)
# shap.summary_plot(shap_values, X_train, max_display=25)
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': np.abs(shap_values).mean(0)
}).sort_values(by='importance', ascending=False)
display(feature_importance[feature_importance['importance'] > 0.000592])
display(feature_importance['importance'].head(71).sum())

,feature,importance
15,GrLivArea,0.081143
3,OverallQual,0.064026
5,YearBuilt,0.039212
4,OverallCond,0.034293
25,GarageCars,0.032058
...,...,...
47,LotShape_IR2,0.001026
60,LotConfig_Inside,0.000993
32,PoolArea,0.000750
72,Neighborhood_Gilbert,0.000691


np.float64(0.6605652567249792)

In [7]:
masker = shap.maskers.Independent(data=X_train)
explainer = shap.TreeExplainer(cat, masker)
shap_values_cat = explainer.shap_values(X = X_train, y = y)
# shap.summary_plot(shap_values, X_train, max_display=25)
feature_importance_cat = pd.DataFrame({
    'feature': X_train.columns,
    'importance': np.abs(shap_values_cat).mean(0)
}).sort_values(by='importance', ascending=False)

 99%|===================| 1452/1460 [00:44<00:00]        

In [93]:
display(feature_importance['importance'].head(70).sum())

np.float64(0.6598150730742481)